In [24]:
# Notebook Processing - Plant Health Analysis
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os
import warnings
warnings.filterwarnings('ignore')

print("="*50)
print("DATA PREPROCESSING AND SPLITTING")
print("="*50)

DATA PREPROCESSING AND SPLITTING


In [25]:
# 1. LOADING THE DATA
print("\n1. LOADING THE DATA")
print("-"*50)
df = pd.read_csv('../data/raw/raw_dataset.csv')
print(f"✓ Data loaded: {df.shape}")
print(f"✓ Columns: {list(df.columns)}")


1. LOADING THE DATA
--------------------------------------------------
✓ Data loaded: (1200, 14)
✓ Columns: ['Timestamp', 'Plant_ID', 'Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature', 'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content', 'Electrochemical_Signal', 'Plant_Health_Status']


In [26]:
# 2. TIMESTAMP PROCESSING
print("\n2. TIMESTAMP PROCESSING")
print("-"*50)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Hour'] = df['Timestamp'].dt.hour
df['Day_of_Week'] = df['Timestamp'].dt.dayofweek
df['Month'] = df['Timestamp'].dt.month
print("✓ Extracted temporal features: Hour, Day_of_Week, Month")


2. TIMESTAMP PROCESSING
--------------------------------------------------
✓ Extracted temporal features: Hour, Day_of_Week, Month


In [27]:
# 3. FEATURE SELECTION
print("\n3. FEATURE SELECTION")
print("-"*50)

feature_columns = [
    'Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature',
    'Humidity', 'Light_Intensity', 'Soil_pH',
    'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level',
    'Chlorophyll_Content', 'Electrochemical_Signal',
    'Hour', 'Day_of_Week', 'Month'
]

X = df[feature_columns].copy()
y = df['Plant_Health_Status'].copy()

print(f"✓ Selected features: {len(feature_columns)}")
print(f"  {feature_columns}")


3. FEATURE SELECTION
--------------------------------------------------
✓ Selected features: 14
  ['Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature', 'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content', 'Electrochemical_Signal', 'Hour', 'Day_of_Week', 'Month']


In [28]:
# 4. HANDLING MISSING VALUES
print("\n4. HANDLING MISSING VALUES")
print("-"*50)

missing_before = X.isnull().sum().sum()
if missing_before > 0:
    print(f"⚠ Missing values detected: {missing_before}")
    X = X.fillna(X.median())
    print("✓ Missing values imputed with median")
else:
    print("✓ No missing values detected")


4. HANDLING MISSING VALUES
--------------------------------------------------
✓ No missing values detected


In [29]:
# 5. TARGET ENCODING (3 CLASSES)
print("\n5. TARGET ENCODING (3 CLASSES)")
print("-"*50)

# Check detected classes
unique_classes = sorted(y.unique())
print(f"Detected classes: {unique_classes}")

# Define fixed class order
class_order = ['Healthy', 'Moderate Stress', 'High Stress']
print(f"Class order: {class_order}")

# Encode target according to our defined order
label_encoder = LabelEncoder()
label_encoder.fit(class_order)
y_encoded = label_encoder.transform(y)

print("\n✓ Encoding summary:")
for idx, class_name in enumerate(label_encoder.classes_):
    count = (y == class_name).sum()
    pct = count / len(y) * 100
    print(f"  {idx} → {class_name}: {count} samples ({pct:.1f}%)")

# Check imbalance
class_counts = pd.Series(y).value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\n✓ Class imbalance ratio: {imbalance_ratio:.2f}")
if imbalance_ratio > 2:
    print("⚠ Warning: Imbalanced classes — stratification will be used during split")


5. TARGET ENCODING (3 CLASSES)
--------------------------------------------------
Detected classes: ['Healthy', 'High Stress', 'Moderate Stress']
Class order: ['Healthy', 'Moderate Stress', 'High Stress']

✓ Encoding summary:
  0 → Healthy: 299 samples (24.9%)
  1 → High Stress: 500 samples (41.7%)
  2 → Moderate Stress: 401 samples (33.4%)

✓ Class imbalance ratio: 1.67


In [30]:
# 6. TRAIN/TEST SPLITTING
print("\n6. TRAIN/TEST SPLITTING")
print("-"*50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"✓ Training set size: {X_train.shape}")
print(f"✓ Test set size: {X_test.shape}")
print(f"✓ Train/Test ratio: {len(X_train)}/{len(X_test)} "
      f"({len(X_train)/len(X)*100:.1f}% / {len(X_test)/len(X)*100:.1f}%)")

# Class distribution in training set
print("\nClass distribution in training set:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  Class {cls} ({label_encoder.classes_[cls]}): {count} ({count/len(y_train)*100:.1f}%)")


6. TRAIN/TEST SPLITTING
--------------------------------------------------
✓ Training set size: (960, 14)
✓ Test set size: (240, 14)
✓ Train/Test ratio: 960/240 (80.0% / 20.0%)

Class distribution in training set:
  Class 0 (Healthy): 239 (24.9%)
  Class 1 (High Stress): 400 (41.7%)
  Class 2 (Moderate Stress): 321 (33.4%)


In [31]:
# 7. FEATURE SCALING
print("\n7. FEATURE SCALING")
print("-"*50)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ StandardScaler applied")
print(f"  Mean of scaled features (train): {X_train_scaled.mean():.4f}")
print(f"  Std of scaled features (train): {X_train_scaled.std():.4f}")


7. FEATURE SCALING
--------------------------------------------------
✓ StandardScaler applied
  Mean of scaled features (train): 0.0000
  Std of scaled features (train): 1.0000


In [32]:
# 8. FINAL DATAFRAMES (NUMERICAL LABELS ONLY)
print("\n8. FINAL DATAFRAMES CREATION")
print("-"*50)

train_df = pd.DataFrame(X_train_scaled, columns=feature_columns)
train_df['Plant_Health_Status'] = y_train

test_df = pd.DataFrame(X_test_scaled, columns=feature_columns)
test_df['Plant_Health_Status'] = y_test

print(f"✓ Training DataFrame: {train_df.shape}")
print(f"✓ Test DataFrame: {test_df.shape}")

print("\n✓ Label encoding check:")
print(f"  Plant_Health_Status values: {train_df['Plant_Health_Status'].unique()}")

print("\n✓ Class mapping:")
for i in range(len(label_encoder.classes_)):
    print(f"  {i} → {label_encoder.classes_[i]}")


8. FINAL DATAFRAMES CREATION
--------------------------------------------------
✓ Training DataFrame: (960, 15)
✓ Test DataFrame: (240, 15)

✓ Label encoding check:
  Plant_Health_Status values: [1 2 0]

✓ Class mapping:
  0 → Healthy
  1 → High Stress
  2 → Moderate Stress


In [33]:
# 9. SAVING PROCESSED DATA
print("\n9. SAVING PROCESSED DATA")
print("-"*50)

os.makedirs('../data/processed', exist_ok=True)

train_df.to_csv('../data/processed/train_data.csv', index=False)
test_df.to_csv('../data/processed/test_data.csv', index=False)

print("✓ Saved: ../data/processed/train_data.csv")
print("✓ Saved: ../data/processed/test_data.csv")

# Save preprocessing objects
import pickle

preprocessing_objects = {
    'scaler': scaler,
    'label_encoder': label_encoder,
    'feature_columns': feature_columns
}

os.makedirs('../model', exist_ok=True)
with open('../model/preprocessing_objects.pkl', 'wb') as f:
    pickle.dump(preprocessing_objects, f)

print("✓ Saved preprocessing objects: ../model/preprocessing_objects.pkl")


9. SAVING PROCESSED DATA
--------------------------------------------------
✓ Saved: ../data/processed/train_data.csv
✓ Saved: ../data/processed/test_data.csv
✓ Saved preprocessing objects: ../model/preprocessing_objects.pkl
